# 🎓 Ciência de Dados — Aula 12: Aprendizado Não Supervisionado 2 (Algoritmos Avançados: Hierárquico, DBSCAN & PCA)
**Professor:** Dr. Rafael Duarte  
**Data:** 29 de Outubro de 2026  
**Formato da Aula:** *Live Coding & Literate Programming* (100% guiada via Notebook)

---

### 🗺️ Onde Estamos no Semestre e Roteiro da Aula 12
* **O que já construímos:**
  * *Aula 09:* Frameworks (CRISP-DM / KDD), Qualidade de Dados e *Data Wrangling* (ETL).
  * *Aula 10:* Preparação Preditiva e **Regressão Linear Simples e Múltipla** (Aprendizado Supervisionado).
  * *Aula 11 (22/10):* **Aprendizado Não Supervisionado 1: Clustering (K-Means)**, Distância Euclidiana, Padronização Z-Score (`StandardScaler`), Cotovelo (*WCSS*), Silhueta e *Personas*.
* **O que faremos HOJE (Aula 12 — 29/10):**
  * **Módulo 1:** Onde o K-Means Falha? As 3 Armadilhas Geométricas (Esfericidade, Obrigatoriedade de Alocar *Outliers* e escolha de $K$ a priori) + Apresentação do Caso Prático (18 Franquias Normais + 2 Anomalias/Fraudes).
  * **Módulo 2:** **Clusterização Hierárquica Aglomerativa e Dendrogramas (Método de Ward)** — Construindo a Árvore Genealógica dos Dados sem precisar adivinhar $K$ antes.
  * **Módulo 3:** **DBSCAN (Agrupamento Baseado em Densidade)** — Encontrando formatos complexos e isolando *Outliers* e Fraudes automaticamente como ruído (`-1`).
  * **Módulo 4:** **Redução de Dimensionalidade com PCA (*Principal Component Analysis*)** — Como comprimir 5 ou 10 variáveis correlacionadas em 2 super-eixos ($PC_1$ e $PC_2$) para visualizar clusters multidimensionais na tela!
  * **☕ Intervalo (15 min)**
  * **Módulo 5:** **Entrega e Validação do Checkpoint 1 (Vale 1,25 pts)** + Laboratório Prático (Aplicando Hierárquico, DBSCAN e PCA na Base do Projeto ou no Dataset da LogiTech Brasil).

---
# 📘 MÓDULO 1: Onde o K-Means Falha? (As 3 Limitações Clássicas)

Na **Aula 11**, vimos que o **K-Means** é rápido, intuitivo e excelente para segmentar dados bem comportados. Porém, por trás da sua simplicidade matemática ($\text{minimizar } \sum \|x_i - \mu_k\|^2$), o K-Means carrega **3 pressupostos rígidos** que frequentemente quebram em bases reais de negócios:

1. **A Ditadura da Esfericidade (Fronteiras de Voronoi):** Como cada ponto simplesmente procura o centroide $\mu_k$ mais próximo em linha reta, o K-Means corta o espaço com retas/planos e assume que todos os grupos são "bolhas esféricas" (*convexas*) de raios parecidos. Se os seus dados tiverem formato alongado, em arco ou em trilhas (ex: rotas logísticas ou padrões de rede), o K-Means corta os grupos ao meio.
2. **A Falta de uma "Lixeira" para Anomalias (Sensibilidade a *Outliers*):** No K-Means, **100% das linhas da tabela são obrigadas a entrar em algum cluster**. Se houver uma fraude financeira ou um erro bizarro no dataset, o K-Means força esse *outlier* para dentro do grupo mais próximo, e — como o centroide $\mu_k$ é uma média aritmética — **o *outlier* puxa o centroide do grupo inteiro para longe da realidade!**
3. **A Exigência de Informar $K$ de Antemão:** O K-Means não sabe se existem 2, 3 ou 7 grupos; ele exige que você defina `n_clusters` antes de dar a partida.

| Algoritmo Não Supervisionado | Precisa definir $K$ antes? | Formato dos Grupos | Como trata *Outliers* / Anomalias? | Melhor Caso de Uso em Negócios |
| :--- | :---: | :---: | :---: | :--- |
| **K-Means** *(Aula 11)* | **Sim** ($K$ fixo) | Apenas esféricos / globulares | **Mal:** Força o outlier em um grupo e distorce a média $\mu_k$. | Segmentação geral de clientes em bases limpas e grandes. |
| **Hierárquico (Ward)** *(Aula 12)* | **Não** (Gera árvore completa) | Esféricos / Hierárquicos | **Médio:** Mostra o outlier isolado num galho alto do Dendrograma. | Taxonomia de produtos, filiais ou clientes quando queremos ver subgrupos. |
| **DBSCAN (Densidade)** *(Aula 12)* | **Não** (Descobre sozinho!) | **Qualquer formato** (curvo, alongado) | **Excelente:** Isola automaticamente como **Ruído (`-1`)**! | Detecção de fraudes, anomalias operacionais e dados geográficos. |

## 1.2. Nosso Cenário Didático: 18 Clientes Normais + 2 Anomalias Extremas ("Café com Dados")

Para vermos na tela do projetor como o **K-Means**, o **Hierárquico** e o **DBSCAN** se comportam diante do perigo, vamos pegar 18 clientes legítimos divididos nas nossas 3 tribos (`Fãs Diários`, `Ocasionais` e `VIPs`), mas agora com **2 intrusos (`Cli_19_Outlier` e `Cli_20_Outlier`)** infiltrados na tabela:
* **`Cli_19_Outlier`:** Renda declarada de **R$ 32.000**, mas visitou a loja apenas **1 vez** no mês.
* **`Cli_20_Outlier`:** Renda declarada de **R$ 150**, mas registrou **55 visitas** no mês (ex: um cartão fidelidade compartilhado indevidamente no balcão / fraude de pontuação!).

Rode a célula abaixo e observe o que acontece quando o K-Means tenta agrupar essa base!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

pd.set_option('display.max_rows', 25)
pd.set_option('display.width', 1000)
plt.rcParams['figure.figsize'] = (14, 5.5)
sns.set_theme(style='whitegrid')

# Base de 20 registros: 18 clientes legítimos (3 tribos de 6) + 2 Outliers extremos!
dados_20 = {
    'ID_Cliente': [
        'Cli_01', 'Cli_02', 'Cli_03', 'Cli_04', 'Cli_05', 'Cli_06',
        'Cli_07', 'Cli_08', 'Cli_09', 'Cli_10', 'Cli_11', 'Cli_12',
        'Cli_13', 'Cli_14', 'Cli_15', 'Cli_16', 'Cli_17', 'Cli_18',
        'Cli_19_Outlier', 'Cli_20_Outlier'
    ],
    'Renda_Mensal_BRL': [
        2300, 2500, 2600, 2800, 2900, 3100,    # Tribo 1: Fãs Diários (Renda ~2.7k, Visitas ~25)
        7400, 7700, 8000, 8300, 8600, 8900,    # Tribo 2: Ocasionais (Renda ~8.1k, Visitas ~3)
        15500, 16000, 16500, 17200, 17800, 18400, # Tribo 3: VIPs (Renda ~16.9k, Visitas ~18)
        32000,                                  # <-- ANOMALIA 1: Bilionário que só foi 1 vez!
        150                                     # <-- ANOMALIA 2: Cartão fraudado (R$ 150 e 55 visitas!)
    ],
    'Visitas_Mensais': [
        24, 26, 25, 27, 23, 25,
        2,  4,  3,  2,  5,  3,
        17, 19, 18, 20, 17, 19,
        1,                                      # <-- ANOMALIA 1
        55                                      # <-- ANOMALIA 2
    ]
}

df_20 = pd.DataFrame(dados_20)

# Padronizando com StandardScaler (Z-Score visto na Aula 11)
scaler_20 = StandardScaler()
X_20_scaled = scaler_20.fit_transform(df_20[['Renda_Mensal_BRL', 'Visitas_Mensais']])

# Rodando o K-Means (K=3) nessa base com 2 outliers:
km_20 = KMeans(n_clusters=3, random_state=42, n_init=10)
df_20['Cluster_KMeans'] = km_20.fit_predict(X_20_scaled)

display(df_20.tail(6))
print("⚠️ Repare acima: O K-Means foi OBRIGADO a colocar o Cli_19_Outlier e o Cli_20_Outlier dentro dos grupos normais,")
print("   contaminando a média salarial e de visitas das tribos legítimas!")

---
# 🌳 MÓDULO 2: Clusterização Hierárquica Aglomerativa e Dendrogramas (Método de Ward)

E se nós não quisermos definir o número $K$ de grupos no escuro? E se quisermos enxergar **toda a hierarquia de parentesco** entre os 20 indivíduos, desde o nível individual até a base inteira?

> 🏛️ **Contexto Histórico & Etimológico:**  
> * **Etimologia:** *Dendrograma* vem do grego *déndron* ($\delta\acute{\varepsilon}\nu\delta\rho o\nu$, "árvore") + *grámma* ($\gamma\rho\alpha\mu\mu\alpha$, "desenho, traçado").  
> * **História:** A taxonomia numérica hierárquica ganhou força na biologia com **Robert Sokal e Peter Sneath (1963)** para agrupar espécies evolutivas. No mesmo ano, o estatístico **Joe H. Ward Jr. (1963)** publicou no *Journal of the American Statistical Association* o seu famoso algoritmo hierárquico baseado na minimização da perda de variância intra-grupo.

## 2.1. Como Funciona a Estratégia Aglomerativa (*Bottom-Up*) e o Critério de Ward?
1. **Início ($n$ grupos):** Cada um dos $20$ clientes começa sozinho em seu próprio grupo individual.
2. **Fusão passo a passo:** O algoritmo procura os dois grupos $A$ e $B$ cuja união provoque o **menor aumento possível na Inércia total ($WCSS$)** e funde ambos em um galho do Dendrograma.
3. **A Equação da Distância de Ward ($\Delta$):**

$$\Delta(A, B) = \frac{n_A \cdot n_B}{n_A + n_B} \|\bar{x}_A - \bar{x}_B\|^2$$

<small>
📖 <b>Guia de Leitura Matemática e Notação (População vs. Amostra):</b><br>
• <b>$\Delta(A, B)$ (Letra grega <i>Delta</i> maiúscula — lê-se <i>"delta de Á, Bê"</i>):</b> Representa o aumento na Soma dos Quadrados Intra-Cluster (custo de variância) caso os subgrupos amostrais $A$ e $B$ sejam fundidos em um único grupo. A altura vertical do "degrau" no Dendrograma é proporcional a esse custo!<br>
• <b>$n_A$ e $n_B$ (Lê-se <i>"êne índice Á"</i> e <i>"êne índice Bê"</i>):</b> Número de observações da <b>Amostra</b> contidas atualmente no subgrupo $A$ e no subgrupo $B$ (lembrando: $N$ maiúsculo designa o tamanho da <b>População</b>, enquanto $n$ minúsculo designa o tamanho da <b>Amostra</b> ou subamostra).<br>
• <b>$\bar{x}_A$ e $\bar{x}_B$ (Lê-se <i>"xis barra índice Á"</i> e <i>"xis barra índice Bê"</i>):</b> Vetores de médias amostrais (baricentros) das variáveis nos grupos $A$ e $B$.<br>
• <b>$\|\bar{x}_A - \bar{x}_B\|^2$ (Lê-se <i>"norma euclidiana ao quadrado de xis barra Á menos xis barra Bê"</i>):</b> Quadrado da distância euclidiana entre os centros dos dois subgrupos.
</small>

Execute a célula abaixo para desenhar o **Dendrograma** dos nossos 20 clientes e veja como os 2 *outliers* (`Cli_19_Outlier` e `Cli_20_Outlier`) aparecem isolados em galhos altos da árvore!

In [ ]:
# Calculando a matriz de ligação hierárquica pelo Método de Ward (scipy.cluster.hierarchy.linkage)
matriz_ward = linkage(X_20_scaled, method='ward')

plt.figure(figsize=(14, 5.5))
dendrogram(
    matriz_ward,
    labels=df_20['ID_Cliente'].values,
    leaf_rotation=45,
    leaf_font_size=10,
    color_threshold=2.2
)
plt.axhline(y=2.2, color='crimson', linestyle='--', lw=2, label='Linha de Corte (Threshold = 2.2)')
plt.title('Dendrograma Hierárquico (Método de Ward) — Veja as 3 Tribos e os 2 Outliers Isolados!')
plt.xlabel('Clientes da Amostra (ID_Cliente)')
plt.ylabel('Distância de Ward (Aumento de Inércia na Fusão)')
plt.legend()
plt.tight_layout()
plt.show()

# Aplicando o AgglomerativeClustering do Scikit-Learn
hc_ward = AgglomerativeClustering(n_clusters=3, linkage='ward')
df_20['Cluster_Hierarquico'] = hc_ward.fit_predict(X_20_scaled)

### 🔍 Como ler o Dendrograma acima?
1. Olhe de baixo para cima: os clientes `Cli_01` a `Cli_06` se juntam rapidinho lá embaixo (distância quase zero, pois são muito parecidos!). O mesmo acontece com `Cli_07` a `Cli_12` e com `Cli_13` a `Cli_18`.
2. Já o **`Cli_19_Outlier`** e o **`Cli_20_Outlier`** demoram muito para conseguir se juntar a qualquer grupo (o "degrau" vertical deles é bem mais alto!).
3. A **Linha Tracejada Vermelha (Corte Horizontal)** funciona como uma tesoura: quantas linhas verticais a tesoura corta naquela altura? Se cortarmos na altura $2,2$, separamos as 3 tribos principais e vemos exatamente quem são os pontos solitários!

---
# 📡 MÓDULO 3: DBSCAN — Agrupamento por Densidade e Caça Automática a Anomalias

Até agora, tanto o K-Means quanto o Hierárquico com `n_clusters=3` acabaram forçando os outliers a entrarem em algum grupo se não fizermos um corte manual.  
Existe um algoritmo que **descobre sozinho quantos grupos existem**, aceita **qualquer formato geométrico** e ainda **separa os *outliers* automaticamente numa lixeira chamada `-1`**?

Sim! Apresento o **DBSCAN** (*Density-Based Spatial Clustering of Applications with Noise*), publicado em **1996** por **Martin Ester, Hans-Peter Kriegel, Jörg Sander e Xiaowei Xu**.

## 3.1. A Matemática da Vizinhança por Densidade ($\varepsilon$ e $MinPts$)
Em vez de calcular centroides globais, o DBSCAN caminha ponto por ponto olhando ao redor com uma "lupa circular" de raio $\varepsilon$ (`eps`):

$$N_{\varepsilon}(x_i) = \{x_q \in X_{\text{amostra}} \mid d(x_i, x_q) \le \varepsilon\}$$

<small>
📖 <b>Guia de Leitura Matemática e Notação:</b><br>
• <b>$\varepsilon$ (Letra grega <i>Épsilon</i> minúscula — lê-se <i>"épsilon"</i>):</b> É o raio máximo da vizinhança ao redor de um ponto (hiperparâmetro <code>eps</code> no Scikit-Learn).<br>
• <b>$N_{\varepsilon}(x_i)$ (Lê-se <i>"êne índice épsilon de xis i"</i>):</b> É o subconjunto de pontos amostrais $x_q$ cuja distância euclidiana $d(x_i, x_q)$ até o ponto $x_i$ é menor ou igual ao raio $\varepsilon$.<br>
• <b>$\text{min\_samples}$ ($MinPts$ — lê-se <i>"mínimo de pontos"</i>):</b> Quantidade mínima de pontos exigida dentro da bola $N_{\varepsilon}(x_i)$ para que aquela região seja considerada "densa" (um cluster legítimo).<br>
• <b>Os 3 Rótulos do DBSCAN:</b><br>
  1. <b>Ponto Central (<i>Core Point</i>):</b> Tem pelo menos <code>min_samples</code> vizinhos dentro do raio $\varepsilon$. Ele funda ou expande um cluster!<br>
  2. <b>Ponto de Borda (<i>Border Point</i>):</b> Fica na beirada do grupo (alcançável por um Ponto Central, mas com menos vizinhos próprios).<br>
  3. <b>Ponto de Ruído / Outlier (<i>Noise</i> = <code>-1</code>):</b> Fica isolado no vazio! O DBSCAN atribui a ele o código especial <b><code>-1</code></b>.
</small>

Vamos rodar o `DBSCAN` na célula abaixo sobre os nossos 20 clientes e comparar o resultado dele com o `K-Means`!

In [ ]:
# Rodando o DBSCAN nos dados padronizados (Z-Score):
# Raio eps = 0.65 desvios-padrão | Densidade mínima = 3 clientes próximos para formar uma tribo
dbscan = DBSCAN(eps=0.65, min_samples=3)
df_20['Cluster_DBSCAN'] = dbscan.fit_predict(X_20_scaled)

print("🎯 Veja como o DBSCAN classificou os 20 clientes (Repare no código -1 nas duas últimas linhas!):")
display(df_20[['ID_Cliente', 'Renda_Mensal_BRL', 'Visitas_Mensais', 'Cluster_KMeans', 'Cluster_DBSCAN']])

# Comparação Visual Lado a Lado: K-Means (Enganado pelos Outliers) vs. DBSCAN (Isola os Outliers como -1!)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Painel 1: K-Means (Centroides puxados pelos outliers)
centroides_km20 = scaler_20.inverse_transform(km_20.cluster_centers_)
sns.scatterplot(
    data=df_20, x='Renda_Mensal_BRL', y='Visitas_Mensais',
    hue='Cluster_KMeans', palette='Set1', s=110, ax=axes[0]
)
axes[0].scatter(centroides_km20[:, 0], centroides_km20[:, 1], color='black', marker='X', s=220, label='Centroides (Deslocados!)')
axes[0].set_title('[ERRO] K-Means (K=3): Força os Outliers nos grupos e entorta os Centroides')
axes[0].set_xlabel('Renda Mensal (R$)')
axes[0].set_ylabel('Visitas Mensais')
axes[0].legend()

# Painel 2: DBSCAN (Identifica as 3 tribos sozinho e marca os 2 Outliers como -1)
df_normais = df_20[df_20['Cluster_DBSCAN'] != -1]
df_anomalias = df_20[df_20['Cluster_DBSCAN'] == -1]

sns.scatterplot(
    data=df_normais, x='Renda_Mensal_BRL', y='Visitas_Mensais',
    hue='Cluster_DBSCAN', palette='viridis', s=110, ax=axes[1]
)
axes[1].scatter(
    df_anomalias['Renda_Mensal_BRL'], df_anomalias['Visitas_Mensais'],
    color='crimson', marker='P', s=240, zorder=5, label='Outliers Detectados (Ruído = -1)'
)
for _, r in df_anomalias.iterrows():
    axes[1].annotate(r['ID_Cliente'], (r['Renda_Mensal_BRL']-3500, r['Visitas_Mensais']-3.5), color='crimson', fontweight='bold')

axes[1].set_title('[OK] DBSCAN (eps=0.65, min_samples=3): Descobre 3 Grupos e Isola as 2 Anomalias (-1)!')
axes[1].set_xlabel('Renda Mensal (R$)')
axes[1].set_ylabel('Visitas Mensais')
axes[1].legend()

plt.tight_layout()
plt.show()

---
# 🔭 MÓDULO 4: Redução de Dimensionalidade com PCA (*Principal Component Analysis*)

Nos Módulos 1 a 3, nossa base didática tinha apenas $m = 2$ colunas (`Renda` e `Visitas`), então era fácil desenhar um gráfico 2D (Eixo X vs. Eixo Y).  
Mas no **Projeto Semestral** de vocês (e na base da *LogiTech Brasil*), vocês agruparão bases com **5, 8 ou 15 colunas numéricas simultâneas**!  
Como o ser humano não enxerga em 5 dimensões (5D), **como podemos desenhar um gráfico 2D na tela que resuma todas as 5 colunas ao mesmo tempo sem perder a separação dos clusters?**

> 🏛️ **Contexto Histórico:**  
> A **Análise de Componentes Principais (PCA)** foi inventada pelo matemático **Karl Pearson (1901)** (o mesmo da correlação $r$!) como um ajuste ortogonal de hiperplanos, e desenvolvida formalmente pelo economista e estatístico **Harold Hotelling (1933)**.

## 4.1. A Álgebra Linear do PCA: Matriz de Covariância, Autovalores e Autovetores
O PCA pega as $m$ variáveis originais padronizadas ($Z$) e encontra novos eixos perpendiculares chamados **Componentes Principais ($PC_1, PC_2, \dots$)** alinhados com a direção de **maior variância (espalhamento)** dos dados:

$$S = \frac{1}{n - 1} Z^T Z \quad \Longrightarrow \quad S \cdot v_k = \lambda_k \cdot v_k \quad \Longrightarrow \quad \text{Variância Explicada}_{k} = \frac{\lambda_k}{\sum_{j=1}^{m} \lambda_j}$$

<small>
📖 <b>Guia de Leitura Matemática e Notação (População vs. Amostra):</b><br>
• <b>$Z$ (Lê-se <i>"zê maiúsculo"</i>):</b> Matriz de dados amostrais padronizados via <code>StandardScaler</code> (com $n$ linhas amostrais e $m$ colunas).<br>
• <b>$Z^T$ (Lê-se <i>"zê transposta"</i>):</b> Matriz $Z$ com linhas e colunas trocadas para multiplicação matricial.<br>
• <b>$S$ vs. $\Sigma$ (Matriz de Covariância Amostral vs. Populacional):</b> $S$ (lê-se <i>"ésse maiúsculo"</i>, dividida pelos graus de liberdade $n-1$) é a matriz de covariância da <b>Amostra</b>; na <b>População</b>, usa-se a letra grega $\Sigma$ (<i>Sigma</i> maiúsculo, dividida por $N$).<br>
• <b>$v_k$ (Lê-se <i>"vê índice ká"</i> — Autovetor ou <i>Eigenvector</i>):</b> Vetor de pesos (cargas fatoriais) que define a direção geométrica do $k$-ésimo Componente Principal ($PC_k$). Cada novo eixo $PC_k$ é uma soma ponderada das variáveis originais!<br>
• <b>$\lambda_k$ (Letra grega <i>Lambda</i> minúscula — lê-se <i>"lâmbda índice ká"</i> — Autovalor ou <i>Eigenvalue</i>):</b> Mede a <b>quantidade de variância (informação)</b> retida pelo eixo $PC_k$. O primeiro eixo ($PC_1$) tem sempre o maior $\lambda_1$; o segundo eixo ($PC_2$) é perpendicular ao primeiro e captura a segunda maior fatia de informação $\lambda_2$, e assim por diante!
</small>

Vamos ver na célula abaixo como o PCA comprime uma base de **5 dimensões (5D)** em apenas **2 dimensões ($PC_1$ e $PC_2$)** preservando quase $90\%$ da informação original!

In [ ]:
# Demonstração do PCA: Comprimindo 5 variáveis logísticas (5D) em 2 Componentes Principais (2D)!
np.random.seed(2026)
n_p = 250

# Gerando as 3 tribos logísticas (Urbana Leve, Interestadual Pesada, Alto Valor/Joias) + 4 Anomalias/Erros!
dist_1, peso_1, tsep_1, vmer_1 = np.random.normal(45, 12, 100).clip(10, 100), np.random.normal(4.5, 1.5, 100).clip(1, 10), np.random.normal(18, 4, 100).clip(8, 35), np.random.normal(180, 40, 100).clip(60, 350)
dist_2, peso_2, tsep_2, vmer_2 = np.random.normal(520, 70, 90).clip(350, 800), np.random.normal(32.0, 4.5, 90).clip(20, 48), np.random.normal(65, 10, 90).clip(40, 95), np.random.normal(1450, 220, 90).clip(900, 2200)
dist_3, peso_3, tsep_3, vmer_3 = np.random.normal(240, 40, 56).clip(130, 340), np.random.normal(2.5, 0.8, 56).clip(0.5, 6), np.random.normal(48, 6, 56).clip(30, 70), np.random.normal(4200, 450, 56).clip(3100, 5500)

# 4 Pedidos Anômalos / Fraudulentos (Ex: distância curtíssima com custo/tempo absurdo ou valor extremo)
dist_out = np.array([15.0, 890.0, 50.0, 780.0])
peso_out = np.array([48.0, 1.0, 45.0, 2.0])
tsep_out = np.array([140.0, 130.0, 5.0, 145.0])
vmer_out = np.array([5800.0, 80.0, 5900.0, 50.0])

distancia_km = np.concatenate([dist_1, dist_2, dist_3, dist_out]).round(1)
peso_kg = np.concatenate([peso_1, peso_2, peso_3, peso_out]).round(2)
tempo_sep = np.concatenate([tsep_1, tsep_2, tsep_3, tsep_out]).round(1)
valor_merc = np.concatenate([vmer_1, vmer_2, vmer_3, vmer_out]).round(2)
custo_entrega = (18.0 + 0.095 * distancia_km + 2.25 * peso_kg + 0.018 * valor_merc + 0.15 * tempo_sep + np.random.normal(0, 4.0, n_p)).round(2)

df_logi_5d = pd.DataFrame({
    'ID_Pedido': [f'PED-{2001+i}' for i in range(n_p)],
    'Distancia_KM': distancia_km,
    'Peso_KG': peso_kg,
    'Tempo_Separacao_Min': tempo_sep,
    'Valor_Mercadoria_BRL': valor_merc,
    'Custo_Operacional_Entrega_BRL': custo_entrega
})

features_5d = ['Distancia_KM', 'Peso_KG', 'Tempo_Separacao_Min', 'Valor_Mercadoria_BRL', 'Custo_Operacional_Entrega_BRL']

# 1. Padronizando as 5 colunas (Matriz Z)
Z_logi = StandardScaler().fit_transform(df_logi_5d[features_5d])

# 2. Aplicando PCA para comprimir de 5D para 2D (PC1 e PC2)
pca = PCA(n_components=2)
coords_pca = pca.fit_transform(Z_logi)
df_logi_5d['PC1'] = coords_pca[:, 0]
df_logi_5d['PC2'] = coords_pca[:, 1]

var_pc1, var_pc2 = pca.explained_variance_ratio_ * 100
print(f"📊 Variância Explicada pelo PC1 (λ1): {var_pc1:.1f}%")
print(f"📊 Variância Explicada pelo PC2 (λ2): {var_pc2:.1f}%")
print(f"✅ Total de Informação das 5 variáveis preservada no gráfico 2D: {var_pc1 + var_pc2:.1f}%!\n")

# Exibindo os Autovetores (Pesos v_k de cada variável original dentro de PC1 e PC2)
df_pesos_pca = pd.DataFrame(pca.components_.T, index=features_5d, columns=['Peso_em_PC1 (v1)', 'Peso_em_PC2 (v2)']).round(3)
print("Composição dos Eixos (Autovetores v1 e v2 — quem puxa cada componente):")
display(df_pesos_pca)

In [ ]:
# Visualizando os Clusters da LogiTech (5D) projetados no Mapa 2D do PCA (K-Means vs. DBSCAN)!
df_logi_5d['Cluster_KMeans'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(Z_logi)
df_logi_5d['Cluster_DBSCAN'] = DBSCAN(eps=0.95, min_samples=6).fit_predict(Z_logi)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Mapa PCA 1: Colorido pelo K-Means
sns.scatterplot(data=df_logi_5d, x='PC1', y='PC2', hue='Cluster_KMeans', palette='Set1', s=70, ax=axes[0])
axes[0].set_title(f'Mapa PCA 2D ({var_pc1+var_pc2:.1f}% da Variância) — Grupos K-Means (Sem filtro de ruído)')
axes[0].set_xlabel(f'Componente Principal 1 - Porte/Peso/Distância ({var_pc1:.1f}%)')
axes[0].set_ylabel(f'Componente Principal 2 - Valor Agregado ({var_pc2:.1f}%)')

# Mapa PCA 2: Colorido pelo DBSCAN (Mostrando os Outliers = -1 em destaque!)
normais_pca = df_logi_5d[df_logi_5d['Cluster_DBSCAN'] != -1]
ruido_pca = df_logi_5d[df_logi_5d['Cluster_DBSCAN'] == -1]

sns.scatterplot(data=normais_pca, x='PC1', y='PC2', hue='Cluster_DBSCAN', palette='viridis', s=70, ax=axes[1])
axes[1].scatter(ruido_pca['PC1'], ruido_pca['PC2'], color='crimson', marker='P', s=180, zorder=5, label=f'Anomalias / Ruído (-1): {len(ruido_pca)} pedidos')
axes[1].set_title(f'Mapa PCA 2D ({var_pc1+var_pc2:.1f}% da Variância) — DBSCAN + Detecção de Anomalias')
axes[1].set_xlabel(f'Componente Principal 1 - Porte/Peso/Distância ({var_pc1:.1f}%)')
axes[1].set_ylabel(f'Componente Principal 2 - Valor Agregado ({var_pc2:.1f}%)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
# ☕ INTERVALO (15 Minutos)
Na volta do intervalo:
* **Módulo 5:** **Fechamento e Entrega do Checkpoint 1 do Projeto (Vale 1,25 pts)** + Laboratório Prático (Aplicando Dendrograma, DBSCAN e PCA na base do grupo ou no desafio da LogiTech Brasil).

---
# 🚀 MÓDULO 5: Consolidação do Checkpoint 1 (1,25 pts) e Laboratório Prático

Com a aula de hoje (**Aula 12 — 29/10**), encerramos o ciclo de **Data Wrangling + Regressão Linear + Aprendizado Não Supervisionado (Clustering)**.  
Na semana que vem (**Aula 13 — 05/11**), entraremos em **Aprendizado Supervisionado: Classificação**, seguido de **Ensembles e Redes Neurais na Aula 14 (12/11)** e **Mentoria Final na Aula 15 (19/11)**.

---

## 5.1. Checklist Oficial de Auditoria do Checkpoint 1 (1,25 Pontos)

O Notebook do Projeto Semestral de cada grupo deve conter, de forma organizada e comentada:

| Bloco do Checkpoint 1 | Técnicas Obrigatórias (Fixação Pedagógica) | Critério de Aceite e Interpretação de Negócio |
| :--- | :--- | :--- |
| **Bloco A: Wrangling (Aula 09)** | `.info()`, `.describe()`, `.drop_duplicates()`, tratamento de `NaN` e filtro de *outliers*. | Base limpa, tipada e documentada segundo o framework de negócio (CRISP-DM). |
| **Bloco B: Regressão (Aula 10)** | `get_dummies(drop_first=True)`, `train_test_split`, imputação pela mediana do treino, `LinearRegression()`. | Apresentar pesos $\beta_j$, métricas no teste ($MAE, RMSE, R^2$) e explicar se o modelo será adotado ou descartado como *baseline*. |
| **Bloco C: Clustering (Aulas 11 e 12)** | `StandardScaler()` (Z-Score), K-Means (Cotovelo + Silhueta) **E** pelo menos 1 técnica avançada de hoje (**Dendrograma de Ward**, **DBSCAN** ou **Mapa 2D com PCA**). | Tabela `.groupby('Cluster').mean()` traduzindo os grupos em **Personas de Negócio** (e mostrando se o DBSCAN/Dendrograma encontrou anomalias `-1` na base do grupo!). |

---

## 5.2. Atividade Prática dos Próximos 40 Minutos (Mão na Massa):

* **OPÇÃO 1 (Grupos com a Base do Projeto aberta):** Apliquem agora na base padronizada (`StandardScaler`) do projeto de vocês o **Dendrograma de Ward**, o **DBSCAN** (para caçar anomalias na base do grupo!) e o **PCA (`n_components=2`)** para plotar o mapa 2D dos clusters do projeto!
* **OPÇÃO 2 (Exercício Guiado na Base `df_logi_5d` da LogiTech Brasil):** Completem as 3 células abaixo usando a base `df_logi_5d` já carregada na memória:
  1. **Célula 1 (Dendrograma):** Plote o Dendrograma com `linkage(Z_logi, method='ward')` e confirme visualmente os 3 grandes ramos logísticos.
  2. **Célula 2 (Caça a Fraudes com DBSCAN):** Liste na tela (`display`) apenas os pedidos que receberam `Cluster_DBSCAN == -1` e explique em comentário por que esses 4 pedidos são anomalias operacionais (olhe a combinação de Distância, Peso e Valor deles!).
  3. **Célula 3 (Tabela Executiva de Personas sem Ruído):** Usando apenas os pedidos normais (`df_logi_5d[df_logi_5d['Cluster_DBSCAN'] != -1]`), gere a tabela `.groupby('Cluster_DBSCAN')[features_5d].mean().round(2)` e dê o nome executivo das 3 categorias de frete!

In [ ]:
# ==============================================================================
# ESPAÇO DO GRUPO — ETAPA 1: DENDROGRAMA HIERÁRQUICO (MÉTODO DE WARD)
# ==============================================================================
# Escreva seu código aqui (na base do seu Projeto ou na matriz Z_logi):

In [ ]:
# ==============================================================================
# ESPAÇO DO GRUPO — ETAPA 2: AUDITORIA DOS OUTLIERS DETECTADOS PELO DBSCAN (-1)
# ==============================================================================
# Escreva seu código aqui para inspecionar quem caiu no cluster -1:

In [ ]:
# ==============================================================================
# ESPAÇO DO GRUPO — ETAPA 3: TABELA EXECUTIVA DE PERSONAS E MAPA PCA 2D
# ==============================================================================
# Escreva seu código aqui: